### Loading the dataset

The data set was generated using the `simulated.py` script

In [1]:
import pandas as pd
import numpy as np

# Loading dataset into dataframe
df = pd.read_csv('../data/simulated_server_metrics.csv', parse_dates=['timestamp'])

df.head()

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0


### Computing the rolling averages
For each row the average of the last N readings are computed for each server. This smooths out noise and shows the recent trends. 

Rolling windows of 5, 10 and 30 are created to capture different times. For example 5 readings = 25 min "very recent", 30 readings = 2.5hrs "much longer".
`min_periods=1` is used for the first 4 rows which would have been `NaN`, this is a tiny fraction of the whole dataset is affected so the issue of less meaningful rows is negliable. 

In [2]:
# Creating rolling windows for the mean and standard deviation
def add_rolling_features(df,column, windows):
    for window in windows:
        mean_col_name = f'{column}_roll_mean_{window}'
        df[mean_col_name] = df.groupby('server_id')[column].rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True)
        std_col_name = f'{column}_roll_standard_deviation_{window}'
        df[std_col_name] = df.groupby('server_id')[column].rolling(window=window, min_periods=1).std().reset_index(level=0, drop=True)
    return df

df = add_rolling_features(df, 'cpu_percent',[5,10,30])
df = add_rolling_features(df, 'memory_percent',[5,10,30])
df = add_rolling_features(df, 'disk_io',[5,10,30])

df.head(30)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,...,memory_percent_roll_mean_10,memory_percent_roll_standard_deviation_10,memory_percent_roll_mean_30,memory_percent_roll_standard_deviation_30,disk_io_roll_mean_5,disk_io_roll_standard_deviation_5,disk_io_roll_mean_10,disk_io_roll_standard_deviation_10,disk_io_roll_mean_30,disk_io_roll_standard_deviation_30
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,22.483571,NaN,22.483571,...,23.274928,NaN,23.274928,NaN,63.451172,NaN,63.451172,NaN,63.451172,NaN
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,20.896125,2.244988,20.896125,...,24.484840,1.711074,24.484840,1.711074,53.716100,13.767470,53.716100,13.767470,53.716100,13.767470
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,21.676897,2.085378,21.676897,...,28.618528,7.261269,28.618528,7.261269,54.841115,9.928172,54.841115,9.928172,54.841115,9.928172
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,23.161460,3.422705,23.161460,...,30.238942,6.756748,30.238942,6.756748,54.036394,8.264545,54.036394,8.264545,54.036394,8.264545
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,22.295015,3.541161,22.295015,...,29.828022,5.923218,29.828022,5.923218,49.958535,11.591881,49.958535,11.591881,49.958535,11.591881
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,21.564164,3.855648,21.717398,...,30.568686,5.599921,30.568686,5.599921,45.431067,9.172940,48.434418,11.019753,48.434418,11.019753
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,23.281641,4.464336,22.600065,...,30.731171,5.130049,30.731171,5.130049,46.335106,9.216942,48.443961,10.059644,48.443961,10.059644
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,23.401387,4.470913,22.754703,...,31.577446,5.318574,31.577446,5.318574,47.141462,10.482656,50.028832,10.336069,50.028832,10.336069
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,21.408883,4.341611,22.187806,...,31.443569,4.991256,31.443569,4.991256,47.744461,10.883881,50.540876,9.789777,50.540876,9.789777
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,22.185596,4.105713,22.240306,...,31.812028,4.847905,31.812028,4.847905,48.680040,9.479579,49.319288,10.005681,49.319288,10.005681


### Computing Z-score

Z-score is the number of standard deviations a data point is from the mean. 

Computed by the following formula :
$$
z = \frac{x - \mu}{\sigma}
$$

`x` = The raw data point,
$\mu$ = The mean ( average )of the dataset,
$\sigma$ = The standard deviation of the set


In [3]:
def compute_z_score(df, column, windows):
    for window in windows:
        column_name = f'{column}_zscore_{window}'
        roll_std_column_name = f'{column}_roll_standard_deviation_{window}'
        roll_mean_column_name = f'{column}_roll_mean_{window}'
        df[column_name] = np.where(df[roll_std_column_name] == 0, 0, (df[column] - df[roll_mean_column_name])/df[roll_std_column_name])
    return df
    
    

df = compute_z_score(df, 'cpu_percent',[5,10,30])
df = compute_z_score(df, 'memory_percent',[5,10,30])
df = compute_z_score(df, 'disk_io',[5,10,30])

df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,...,disk_io_roll_standard_deviation_30,cpu_percent_zscore_5,cpu_percent_zscore_10,cpu_percent_zscore_30,memory_percent_zscore_5,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,22.483571,NaN,22.483571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,20.896125,2.244988,20.896125,...,13.767470,-0.707107,-0.707107,-0.707107,0.707107,0.707107,0.707107,-0.707107,-0.707107,-0.707107
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,21.676897,2.085378,21.676897,...,9.928172,0.748807,0.748807,0.748807,1.138558,1.138558,1.138558,0.226631,0.226631,0.226631
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,23.161460,3.422705,23.161460,...,8.264545,1.301219,1.301219,1.301219,0.719465,0.719465,0.719465,-0.292111,-0.292111,-0.292111
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,22.295015,3.541161,22.295015,...,11.591881,-0.978713,-0.978713,-0.978713,-0.277498,-0.277498,-0.277498,-1.407143,-1.407143,-1.407143
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,21.564164,3.855648,21.717398,...,11.019753,-0.709310,-0.832549,-0.832549,0.465611,0.661317,0.661317,-0.503354,-0.691538,-0.691538
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,23.281641,4.464336,22.600065,...,10.059644,1.033619,1.345976,1.345976,-0.450617,0.190039,0.190039,0.235014,0.005692,0.005692
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,23.401387,4.470913,22.754703,...,10.336069,0.097471,0.295033,0.295033,1.167183,1.113819,1.113819,1.333771,1.073338,1.073338
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,21.408883,4.341611,22.187806,...,9.789777,-0.865175,-1.184031,-1.184031,-0.564917,-0.214579,-0.214579,0.633301,0.418432,0.418432
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,22.185596,4.105713,22.240306,...,10.005681,0.128407,0.130702,0.130702,0.472317,0.684034,0.684034,-1.092353,-1.098805,-1.098805


### Computing time-based features

In [4]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,...,cpu_percent_zscore_10,cpu_percent_zscore_30,memory_percent_zscore_5,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30,hour,day_of_week
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,22.483571,NaN,22.483571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,20.896125,2.244988,20.896125,...,-0.707107,-0.707107,0.707107,0.707107,0.707107,-0.707107,-0.707107,-0.707107,0,2
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,21.676897,2.085378,21.676897,...,0.748807,0.748807,1.138558,1.138558,1.138558,0.226631,0.226631,0.226631,0,2
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,23.161460,3.422705,23.161460,...,1.301219,1.301219,0.719465,0.719465,0.719465,-0.292111,-0.292111,-0.292111,0,2
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,22.295015,3.541161,22.295015,...,-0.978713,-0.978713,-0.277498,-0.277498,-0.277498,-1.407143,-1.407143,-1.407143,0,2
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,21.564164,3.855648,21.717398,...,-0.832549,-0.832549,0.465611,0.661317,0.661317,-0.503354,-0.691538,-0.691538,0,2
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,23.281641,4.464336,22.600065,...,1.345976,1.345976,-0.450617,0.190039,0.190039,0.235014,0.005692,0.005692,0,2
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,23.401387,4.470913,22.754703,...,0.295033,0.295033,1.167183,1.113819,1.113819,1.333771,1.073338,1.073338,0,2
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,21.408883,4.341611,22.187806,...,-1.184031,-1.184031,-0.564917,-0.214579,-0.214579,0.633301,0.418432,0.418432,0,2
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,22.185596,4.105713,22.240306,...,0.130702,0.130702,0.472317,0.684034,0.684034,-1.092353,-1.098805,-1.098805,0,2


### Rate of change 

Essentially calculating how much did this value change from the previous reading for each `server_id`.

In [5]:
def calculate_rate_of_change(df,column):
    column_name = f'{column}_rate_of_change'
    df[column_name] = df.groupby('server_id')[column].diff()
    return df

df = calculate_rate_of_change(df,'cpu_percent')
df = calculate_rate_of_change(df,'memory_percent')
df = calculate_rate_of_change(df,'disk_io')
df.head(10)

,timestamp,server_id,server_type,cpu_percent,memory_percent,disk_io,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,...,memory_percent_zscore_10,memory_percent_zscore_30,disk_io_zscore_5,disk_io_zscore_10,disk_io_zscore_30,hour,day_of_week,cpu_percent_rate_of_change,memory_percent_rate_of_change,disk_io_rate_of_change
0,2025-01-01 00:00:00,web_1,web,22.483571,23.274928,63.451172,0,22.483571,NaN,22.483571,...,NaN,NaN,NaN,NaN,NaN,0,2,NaN,NaN,NaN
1,2025-01-01 00:05:00,web_1,web,19.308678,25.694752,43.981028,0,20.896125,2.244988,20.896125,...,0.707107,0.707107,-0.707107,-0.707107,-0.707107,0,2,-3.174892,2.419824,-19.470143
2,2025-01-01 00:10:00,web_1,web,23.238443,36.885905,57.091144,0,21.676897,2.085378,21.676897,...,1.138558,1.138558,0.226631,0.226631,0.226631,0,2,3.929764,11.191153,13.110116
3,2025-01-01 00:15:00,web_1,web,27.615149,35.100183,51.622232,0,23.161460,3.422705,23.161460,...,0.719465,0.719465,-0.292111,-0.292111,-0.292111,0,2,4.376707,-1.785722,-5.468912
4,2025-01-01 00:20:00,web_1,web,18.829233,28.184342,33.647097,0,22.295015,3.541161,22.295015,...,-0.277498,-0.277498,-1.407143,-1.407143,-1.407143,0,2,-8.785916,-6.915841,-17.975135
5,2025-01-01 00:25:00,web_1,web,18.829315,34.272010,40.813835,0,21.564164,3.855648,21.717398,...,0.661317,0.661317,-0.503354,-0.691538,-0.691538,0,2,0.000082,6.087668,7.166738
6,2025-01-01 00:30:00,web_1,web,27.896064,31.706079,48.501220,0,23.281641,4.464336,22.600065,...,0.190039,0.190039,0.235014,0.005692,0.005692,0,2,9.066749,-2.565931,7.687384
7,2025-01-01 00:35:00,web_1,web,23.837174,37.501373,61.122924,0,23.401387,4.470913,22.754703,...,1.113819,1.113819,1.333771,1.073338,1.073338,0,2,-4.058890,5.795294,12.621705
8,2025-01-01 00:40:00,web_1,web,17.652628,30.372548,54.637231,0,21.408883,4.341611,22.187806,...,-0.214579,-0.214579,0.633301,0.418432,0.418432,0,2,-6.184546,-7.128825,-6.485694
9,2025-01-01 00:45:00,web_1,web,22.712800,35.128161,38.324992,0,22.185596,4.105713,22.240306,...,0.684034,0.684034,-1.092353,-1.098805,-1.098805,0,2,5.060172,4.755613,-16.312239


### Remove NaN values
Standard deviation is calculated by 

$$s = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$$

and requires atleast 2 values to be computed, we have 15 servers, exactly 15 first rows and 15 NaN's consistently across the `std` and `Z-score`. Below we drop these rows.

We also have NaN values the `cpu_percent_rate_of_change`, `memory_percent_rate_of_change` and `disk_io_rate_of_change`, on the first row for every `server_id`

In [6]:
df.isna().sum()

timestamp                                     0
server_id                                     0
server_type                                   0
cpu_percent                                   0
memory_percent                                0
disk_io                                       0
is_anomaly                                    0
cpu_percent_roll_mean_5                       0
cpu_percent_roll_standard_deviation_5        15
cpu_percent_roll_mean_10                      0
cpu_percent_roll_standard_deviation_10       15
cpu_percent_roll_mean_30                      0
cpu_percent_roll_standard_deviation_30       15
memory_percent_roll_mean_5                    0
memory_percent_roll_standard_deviation_5     15
memory_percent_roll_mean_10                   0
memory_percent_roll_standard_deviation_10    15
memory_percent_roll_mean_30                   0
memory_percent_roll_standard_deviation_30    15
disk_io_roll_mean_5                           0
disk_io_roll_standard_deviation_5       

In [7]:
print(len(df))
df = df.dropna()
print(len(df))
df.isna().sum()

90720
90705


timestamp                                    0
server_id                                    0
server_type                                  0
cpu_percent                                  0
memory_percent                               0
disk_io                                      0
is_anomaly                                   0
cpu_percent_roll_mean_5                      0
cpu_percent_roll_standard_deviation_5        0
cpu_percent_roll_mean_10                     0
cpu_percent_roll_standard_deviation_10       0
cpu_percent_roll_mean_30                     0
cpu_percent_roll_standard_deviation_30       0
memory_percent_roll_mean_5                   0
memory_percent_roll_standard_deviation_5     0
memory_percent_roll_mean_10                  0
memory_percent_roll_standard_deviation_10    0
memory_percent_roll_mean_30                  0
memory_percent_roll_standard_deviation_30    0
disk_io_roll_mean_5                          0
disk_io_roll_standard_deviation_5            0
disk_io_roll_

### Adding server type
This would be an encoded categorical feature, this will allow the model to account for type differences.

In [8]:
df = pd.get_dummies(df,columns=['server_type'],dtype=int)
df.head()

,timestamp,server_id,cpu_percent,memory_percent,disk_io,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,cpu_percent_roll_standard_deviation_10,...,hour,day_of_week,cpu_percent_rate_of_change,memory_percent_rate_of_change,disk_io_rate_of_change,server_type_batch_worker,server_type_cache,server_type_database,server_type_load_balancer,server_type_web
1,2025-01-01 00:05:00,web_1,19.308678,25.694752,43.981028,0,20.896125,2.244988,20.896125,2.244988,...,0,2,-3.174892,2.419824,-19.470143,0,0,0,0,1
2,2025-01-01 00:10:00,web_1,23.238443,36.885905,57.091144,0,21.676897,2.085378,21.676897,2.085378,...,0,2,3.929764,11.191153,13.110116,0,0,0,0,1
3,2025-01-01 00:15:00,web_1,27.615149,35.100183,51.622232,0,23.161460,3.422705,23.161460,3.422705,...,0,2,4.376707,-1.785722,-5.468912,0,0,0,0,1
4,2025-01-01 00:20:00,web_1,18.829233,28.184342,33.647097,0,22.295015,3.541161,22.295015,3.541161,...,0,2,-8.785916,-6.915841,-17.975135,0,0,0,0,1
5,2025-01-01 00:25:00,web_1,18.829315,34.272010,40.813835,0,21.564164,3.855648,21.717398,3.468963,...,0,2,0.000082,6.087668,7.166738,0,0,0,0,1


In [12]:
print(df.columns)
df.head()

Index(['timestamp', 'server_id', 'cpu_percent', 'memory_percent', 'disk_io',
       'is_anomaly', 'cpu_percent_roll_mean_5',
       'cpu_percent_roll_standard_deviation_5', 'cpu_percent_roll_mean_10',
       'cpu_percent_roll_standard_deviation_10', 'cpu_percent_roll_mean_30',
       'cpu_percent_roll_standard_deviation_30', 'memory_percent_roll_mean_5',
       'memory_percent_roll_standard_deviation_5',
       'memory_percent_roll_mean_10',
       'memory_percent_roll_standard_deviation_10',
       'memory_percent_roll_mean_30',
       'memory_percent_roll_standard_deviation_30', 'disk_io_roll_mean_5',
       'disk_io_roll_standard_deviation_5', 'disk_io_roll_mean_10',
       'disk_io_roll_standard_deviation_10', 'disk_io_roll_mean_30',
       'disk_io_roll_standard_deviation_30', 'cpu_percent_zscore_5',
       'cpu_percent_zscore_10', 'cpu_percent_zscore_30',
       'memory_percent_zscore_5', 'memory_percent_zscore_10',
       'memory_percent_zscore_30', 'disk_io_zscore_5', 'disk_io_

,timestamp,server_id,cpu_percent,memory_percent,disk_io,is_anomaly,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_roll_mean_10,cpu_percent_roll_standard_deviation_10,...,hour,day_of_week,cpu_percent_rate_of_change,memory_percent_rate_of_change,disk_io_rate_of_change,server_type_batch_worker,server_type_cache,server_type_database,server_type_load_balancer,server_type_web
1,2025-01-01 00:05:00,web_1,19.308678,25.694752,43.981028,0,20.896125,2.244988,20.896125,2.244988,...,0,2,-3.174892,2.419824,-19.470143,0,0,0,0,1
2,2025-01-01 00:10:00,web_1,23.238443,36.885905,57.091144,0,21.676897,2.085378,21.676897,2.085378,...,0,2,3.929764,11.191153,13.110116,0,0,0,0,1
3,2025-01-01 00:15:00,web_1,27.615149,35.100183,51.622232,0,23.161460,3.422705,23.161460,3.422705,...,0,2,4.376707,-1.785722,-5.468912,0,0,0,0,1
4,2025-01-01 00:20:00,web_1,18.829233,28.184342,33.647097,0,22.295015,3.541161,22.295015,3.541161,...,0,2,-8.785916,-6.915841,-17.975135,0,0,0,0,1
5,2025-01-01 00:25:00,web_1,18.829315,34.272010,40.813835,0,21.564164,3.855648,21.717398,3.468963,...,0,2,0.000082,6.087668,7.166738,0,0,0,0,1


### Rolling window limitation: sustained anomalies get absorbed into their own baseline

When comparing `cpu_zscore_5` against `cpu_zscore_30` for `web_1`'s injected 2-hour CPU spike
(2025-01-10, 10:00–12:00), a clear limitation of rolling-window-based Z-scores becomes visible.

With a short window (5), the Z-score is highest at the very start of the anomaly (~1.5), but
drops toward zero — and even goes negative — within about 25 minutes. This happens because the
rolling mean/std are recalculated from the last 5 readings, which very quickly become entirely
made up of anomalous values. The window "absorbs" the anomaly and starts treating it as the new
normal, even though the underlying issue is still ongoing.

A longer window (30) is more resistant to this — the Z-score stays positive and clearly elevated
(roughly 0.3–2.6) for the full 2-hour window, since the anomaly can't fully saturate 30 readings
within that time. However, even this longer window shows a gradual decline in Z-score over the
duration of the anomaly, as more anomalous readings enter the window.

**Takeaway:** short rolling windows are effective at flagging the *onset* of a sustained anomaly,
but rapidly lose sensitivity the longer that anomaly continues, because the baseline they compare
against is itself built from recent — and increasingly contaminated — data. Longer windows delay
this effect but don't eliminate it. Real-world mitigations exist (e.g. freezing the baseline once
an anomaly is flagged, or comparing against a longer, historically separate reference window), but
were considered out of scope for this project; the limitation is instead documented here and
factored into how detection thresholds are chosen.

In [9]:
df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)][['timestamp', 'cpu_percent', 'cpu_percent_roll_mean_30', 'cpu_percent_roll_standard_deviation_30', 'cpu_percent_zscore_30']]

,timestamp,cpu_percent,cpu_percent_roll_mean_30,cpu_percent_roll_standard_deviation_30,cpu_percent_zscore_30
2712,2025-01-10 10:00:00,93.206469,37.697389,21.285964,2.607779
2713,2025-01-10 10:05:00,95.118247,40.079352,23.539806,2.338120
2714,2025-01-10 10:10:00,86.219212,42.230816,24.719715,1.779486
2715,2025-01-10 10:15:00,92.788924,44.604421,26.051370,1.849596
2716,2025-01-10 10:20:00,90.476344,46.875421,26.996571,1.615054
2717,2025-01-10 10:25:00,97.761290,49.723392,27.722607,1.732806
2718,2025-01-10 10:30:00,92.199874,52.003924,28.322720,1.419212
2719,2025-01-10 10:35:00,96.933005,54.466279,28.924532,1.468191
2720,2025-01-10 10:40:00,97.395977,57.215592,28.955523,1.387659
2721,2025-01-10 10:45:00,97.640736,59.844297,28.925476,1.306683


In [10]:
df[(df['server_id'] == 'web_1') & (df['is_anomaly'] == 1)][['timestamp', 'cpu_percent', 'cpu_percent_roll_mean_5', 'cpu_percent_roll_standard_deviation_5', 'cpu_percent_zscore_5']]

,timestamp,cpu_percent,cpu_percent_roll_mean_5,cpu_percent_roll_standard_deviation_5,cpu_percent_zscore_5
2712,2025-01-10 10:00:00,93.206469,62.493170,20.279148,1.514526
2713,2025-01-10 10:05:00,95.118247,72.219648,22.249243,1.029186
2714,2025-01-10 10:10:00,86.219212,76.336664,22.627333,0.436753
2715,2025-01-10 10:15:00,92.788924,86.531306,12.321851,0.507847
2716,2025-01-10 10:20:00,90.476344,91.561839,3.412497,-0.318094
2717,2025-01-10 10:25:00,97.761290,92.472804,4.420397,1.196383
2718,2025-01-10 10:30:00,92.199874,91.889129,4.169304,0.074532
2719,2025-01-10 10:35:00,96.933005,94.031887,3.157057,0.918931
2720,2025-01-10 10:40:00,97.395977,94.953298,3.368798,0.725089
2721,2025-01-10 10:45:00,97.640736,96.386176,2.361604,0.531232


### Save the new dataset

In [11]:
df.to_csv('../data/features.csv', index=False)